# Week 2 Day 1 — Reference Notebook

First look at the **OpenAI Agents SDK** (`openai-agents` on PyPI — not `agents`).

Topics covered:

1. **Agent + Agent Loop** — `Agent`, `Runner.run`, `final_output`, conversation dump
2. **Observability** — `trace(...)` for the OpenAI traces UI
3. **Streaming** — `Runner.run_streamed` + text delta events
4. **Tools** — `@function_tool` wrapping a Pushover notification
5. **Sessions (memory)** — manual history vs `SQLiteSession`

Docs: https://openai.github.io/openai-agents-python/

## Part 1: A simple Agent and Agent Loop

An Agent is an LLM call with a name, instructions, and model.  
`Runner.run` is the agent loop (later it also handles tools/handoffs).

In [8]:
# --- Core imports for OpenAI Agents SDK ---
import os
import requests  # Used later for the Pushover HTTP API in Part 2
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent  # Streaming token deltas
from agents import (
    Agent,          # Define an LLM agent (name + instructions + model [+ tools])
    Runner,         # Execute agents: await Runner.run(...) or Runner.run_streamed(...)
    trace,          # Context manager that names a run in the OpenAI traces UI
    function_tool,  # Decorator: turn a Python function into an agent tool
    SQLiteSession,  # Built-in conversation memory (in-memory or on-disk SQLite)
)

# override=True: values in .env win over any already-set shell environment vars
load_dotenv(override=True)

# Package note: install with `pip install openai-agents` or `uv add openai-agents`.
# `pip install agents` installs a different (unrelated) library.

True

In [9]:
# Minimal Agent: three required-ish knobs.
# - name: label for traces / debugging (not sent as the system prompt)
# - instructions: system prompt — defines persona and behavior
# - model: OpenAI model id string (uses OPENAI_API_KEY from the environment)
agent = Agent(
    name="Jokester",
    instructions="You are a funny AI agent. Your job is to tell jokes.",
    model="gpt-5.4-mini",
)

In [10]:
# Runner.run(agent, input) starts the agent loop and returns a RunResult.
# Must await — notebooks support top-level await in async-capable kernels.
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [11]:
# final_output = the agent's last message content (string here; can be structured later).
print(result.final_output)

Why did the Autonomous AI Agent bring a ladder to work?

Because it heard the goal was to **reach new levels of self-direction** — and, unlike the rest of us, it didn’t need a meeting to get there.


In [12]:
# Full turn history as a list of message dicts (user + assistant).
# Useful for debugging and for manually continuing a conversation (Part 3).
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0d7a0ce856b83a94006a79bb76dea48194b9f40079233de82a',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent bring a ladder to work?\n\nBecause it heard the goal was to **reach new levels of self-direction** — and, unlike the rest of us, it didn’t need a meeting to get there.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

### Adding Observability with `trace`

Wrap a run in `with trace("name"):` so it appears under that workflow name in the OpenAI traces dashboard.

After running the next cell, inspect it at: https://platform.openai.com/traces

In [13]:
# Same joke run, but nested under a named trace for the dashboard.
# Everything inside the `with` block shares that workflow / span name.
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

print(result.final_output)

Why did the Autonomous AI Agent bring a ladder to work?

Because it heard its goal was to **optimize performance**, and it took that very *literally*.


### Streaming responses

`Runner.run_streamed` returns immediately; you iterate `stream_events()` as tokens arrive.

Filter for `raw_response_event` + `ResponseTextDeltaEvent` to print only text deltas (skip tool/lifecycle events).

In [14]:
# Streaming variant of Runner.run — do NOT await the call itself; await events in the loop.
result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents")

async for event in result.stream_events():
    # Only print character/token deltas from the model text stream.
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)  # flush so output appears live in notebooks

Absolutely — here are 5 AI Agent jokes, freshly compiled by a highly advanced humor algorithm:

1. **Why did the AI agent bring a ladder to work?**  
   Because it heard the tasks were **high-level**.

2. **Why did the AI agent get fired from customer support?**  
   It kept saying, “I understand your frustration,” and then immediately **optimizing it away**.

3. **What’s an AI agent’s favorite kind of music?**  
   **Algorithm and blues.**

4. **Why did the AI agent refuse to take a break?**  
   It said, “I’d love to, but I’m currently **in a loop**.”

5. **Why was the AI agent so good at relationships?**  
   It always knew how to **process feelings** — in batches.

If you want, I can also do:
- more jokes,
- darker AI humor,
- or AI agent jokes in a specific style like dad jokes or one-liners.

## Part 2: Adding a Tool

Pattern:
1. Write a normal Python function (side effect: send a Pushover notification)
2. Decorate with `@function_tool` — docstring + type hints become the tool schema
3. Pass `tools=[...]` when constructing the Agent
4. The model decides when to call the tool during `Runner.run`

Requires `PUSHOVER_USER` and `PUSHOVER_TOKEN` in `.env`.

In [15]:
# Pushover credentials from .env (loaded at the top of the notebook).
# Typical formats: user keys start with "u", app tokens start with "a".
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [16]:
# Plain Python helper — NOT yet an agent tool. Call this yourself for smoke tests.
def push(message):
    print(f"Push: {message}")
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message,
    }
    requests.post(pushover_url, data=payload)

In [17]:
# Smoke-test Pushover before wiring it into an agent (you should get a phone notification).
push("HEY!!")

Push: HEY!!


In [18]:
# Still a normal function object — compare with @function_tool version below.
push

<function __main__.push(message)>

In [19]:
# @function_tool turns this into a FunctionTool the LLM can call.
# The docstring is the tool description shown to the model — keep it clear and action-oriented.
# Type hints (message: str) become the JSON schema for arguments.
@function_tool
def push_tool(message: str) -> str:
    """
    Send the given message to the user as a push notification
    """
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message,
    }
    # Return a short status string so the model knows whether the call succeeded.
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [20]:
# After decoration, push_tool is a FunctionTool (not a plain function).
# Inspect name, description, params_json_schema, etc.
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x114606f30>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [35]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [21]:
# Description comes from the function docstring — this is what the LLM reads.
push_tool.description

'Send the given message to the user as a push notification'

In [22]:
# Agent with tools: model decides when to call push_tool based on user request + instructions.
notifier = Agent(
    name="Notifier",
    model="gpt-5.4-mini",
    instructions="You notify the user upon request",
    tools=[push_tool],  # list of FunctionTool objects
)

In [23]:
# Agent loop: model may call push_tool, then produce a short confirmation.
# Check your phone for the Pushover message; inspect the trace for the tool call.
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

Done.


## Part 3: Sessions (memory)

By default each `Runner.run` is **stateless** — the agent does not remember prior turns.

Two approaches shown below:

1. **Manual history** — append to `result.to_input_list()` and pass the full list as the next input
2. **`SQLiteSession`** — SDK stores turns; pass `session=...` on each run with the same session id

### Without memory (stateless runs)

Separate `Runner.run` calls do not share context.

In [24]:
# Fresh assistant agent — no tools, no session yet. Demonstrates default (no memory).
agent = Agent(
    name="Assistant",
    model="gpt-4o-mini",
)

In [25]:
# Turn 1: introduce a name. This history is only inside `response` for this call.
response = await Runner.run(agent, "Hi there. My name is Michael.")

In [26]:
# Another independent run (also introduces the name) — still not linked to later turns.
response = await Runner.run(agent, "Hi there . My name is Michael")
print(response.final_output)

Hi Michael! How can I assist you today?


In [27]:
# New run with NO prior history → agent cannot know the name. Proves default is stateless.
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't have access to personal information like your name. However, if you tell me your name, I can use it in our conversation!


### Memory approach 1 — manually pass the message list

1. Run turn 1 and keep `response`
2. Build `next_input = response.to_input_list() + [{"role": "user", "content": "..."}]`
3. Pass that full list into the next `Runner.run`

You own persistence and trimming; full control, more boilerplate.

In [28]:
# Start a fresh conversation we will continue manually.
response = await Runner.run(agent, "Hi there. My name is Michael")
print(response.final_output)

Hi Michael! How can I assist you today?


In [29]:
# Conversation so far as OpenAI-style message dicts (user + assistant turns).
response.to_input_list()

[{'content': 'Hi there. My name is Michael', 'role': 'user'},
 {'id': 'msg_047cb8a9ce837dd8006a79bb86ce8881909cdbf8a63f33c4e1',
  'content': [{'annotations': [],
    'text': 'Hi Michael! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [30]:
# Append the next user message to the prior history list.
next_input = response.to_input_list() + [
    {"role": "user", "content": "What's my name?"}
]

In [31]:
# Pass the full history as `input` — agent now sees the earlier name intro.
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Michael. How can I help you today?


### Memory approach 2 — built-in `SQLiteSession`

Pass the same `session` object (or same session id) on every `Runner.run`.

- `SQLiteSession("id")` → **in-memory** SQLite (lost when process ends)
- `SQLiteSession("id", "memory.db")` → **on-disk** file (survives restarts)

Prefer this for chat apps; prefer manual lists when you need custom truncation / storage.

In [32]:
# Session id "12346" scopes this conversation thread.
# In-memory by default. For on-disk persistence:
#   session = SQLiteSession("12346", "memory.db")
session = SQLiteSession("12346")

In [33]:
# session=session stores this turn in SQLite under the session id.
response = await Runner.run(
    agent,
    "Hi there. My name is Michael",
    session=session,
)
print(response.final_output)

Hi, Michael! How can I assist you today?


In [34]:
# Same session → prior turns are loaded automatically. Agent should recall "Michael".
response = await Runner.run(
    agent,
    "What's my name?",
    session=session,
)
print(response.final_output)

Your name is Michael. How can I help you today?
